# CommissionLens — Full Pipeline Notebook

**Commission-Adjusted Alpha Prediction in Indian Mutual Funds**

Run each cell in order. All heavy lifting is in the `src/` modules — this notebook orchestrates and visualises.

## 1. Setup

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Image, display

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

from config import *
print(f"Date range: {START_DATE} → {END_DATE}")
print(f"Seed funds: {len(SEED_FUNDS)}")

## 2. Data Collection

In [ ]:
from src.data_collection import collect_all
nav_df, bench_df, macro_df = collect_all()
print(f"NAV: {nav_df.shape}, Benchmark: {bench_df.shape}, Macro: {macro_df.shape}")

In [ ]:
# Regular vs Direct NAV — the gap IS the commission
fund = nav_df['fund_name'].unique()[0]
s = nav_df[nav_df['fund_name'] == fund]
fig, ax = plt.subplots()
ax.plot(s['date'], s['nav_regular'], label='Regular')
ax.plot(s['date'], s['nav_direct'], label='Direct')
ax.fill_between(s['date'], s['nav_regular'], s['nav_direct'], alpha=0.15, color='red')
ax.set_title(f'{fund} — Commission Gap'); ax.legend()
plt.tight_layout(); plt.show()

## 3. Feature Engineering

In [ ]:
from src.feature_engineering import engineer_features
features_df = engineer_features(nav_df, bench_df)
print(f"Feature matrix: {features_df.shape}")
features_df.head()

## 4. Target Building

In [ ]:
from src.target_builder import build_targets
dataset_df = build_targets(features_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
dataset_df['target_justified'].value_counts().plot.pie(
    ax=axes[0], autopct='%1.1f%%', labels=['Unjustified', 'Justified'],
    colors=['#E74C3C', '#2ECC71'])
axes[0].set_title('Commission Justification Split'); axes[0].set_ylabel('')
dataset_df['target_net_alpha'].clip(-0.1, 0.1).apply(lambda x:x*100).hist(
    bins=40, ax=axes[1], color='#3498DB', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Net Alpha Distribution (%)'); plt.tight_layout(); plt.show()

## 5. DNN Training

In [ ]:
from src.model_training import train_all_models
metrics, (model, scaler, feat_cols) = train_all_models(dataset_df)
print("\n" + json.dumps(metrics, indent=2))

## 6. SHAP Explainability

In [ ]:
from src.shap_analysis import run_shap_analysis
fi = run_shap_analysis(dataset_df)

In [ ]:
display(Image("../reports/shap_summary.png", width=700))

In [ ]:
display(Image("../reports/shap_bar.png", width=700))

## 7. SIP Back-Validation

In [ ]:
from src.sip_simulation import run_sip_backtest
sip_df, summary = run_sip_backtest(dataset_df)
print(summary.to_string())

In [ ]:
display(Image("../reports/sip_comparison.png", width=700))

## 8. Done

All deliverables generated:
- `data/fund_dataset.csv`
- `models/dnn_model.pt`, `dnn_scaler.pkl`, `dnn_arch.pkl`
- `reports/metrics.json`, `shap_summary.png`, `shap_bar.png`, `shap_report.txt`, `sip_comparison.png`

Run `streamlit run app.py` from the project root for the interactive dashboard.